In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("geotech_kmeans_homework.ipynb")

# Geotechnical and Geoenvironmental Engineering: Clustering Compaction Tests with K-means

> **Mohamad M. Hallal, PhD** <br> Teaching Professor, UC Berkeley

[![License](https://img.shields.io/badge/license-CC%20BY--NC--ND%204.0-blue)](https://creativecommons.org/licenses/by-nc-nd/4.0/)
***

## About this Assignment

A regional geotechnical laboratory recently conducted Proctor compaction tests on **36 soil specimens** collected from various sites. Unfortunately, the spreadsheet linking each test to its original soil description was corrupted, and only the raw compaction data and index properties remain.

Your task is to use **unsupervised machine learning** — specifically K-means clustering via SciPy — to recover likely soil groupings from the compaction results. You will:

1. Fit polynomial curves to the raw Proctor data and extract optimum moisture content (OMC) and maximum dry density (MDD).
2. Assemble those quantities into a feature matrix alongside grain-size and Atterberg limit data.
3. Cluster the tests with `scipy.cluster.vq.kmeans2`.
4. Use geotechnical reasoning to identify which cluster corresponds to **sandy**, **silty**, and **clayey** soil families.

This assignment builds directly on the Proctor compaction concepts from the *Soil Compaction* notebook. Review that notebook if you need a refresher on OMC and MDD.

In [ ]:
# Please run this cell, and do not modify the contents

import hashlib
def get_hash(num):
    """Helper function for assessing correctness"""
    return hashlib.md5(str(num).encode()).hexdigest()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.vq import whiten, kmeans2, vq

plt.rcParams["figure.figsize"] = (7, 5)
pd.set_option("display.max_columns", 20)

## Autograding and Gradescope

This assignment uses an automatic grader. Each question has an **Answer Cell** followed by a **Testing Cell**. Place your code in the Answer Cell as instructed, then run the Testing Cell to check your work. The output of the Testing Cell shows whether your answer is correct.

> **Do not rename the answer variables** (`q1`, `q2`, etc.) — the grader looks for those specific names.

## Background: Proctor Test and Soil-Type Differences

The **Standard Proctor compaction test** (ASTM D698) determines how dry unit weight varies with water content for a given soil. Soil is compacted in a standardized mold at several water contents, and the dry unit weight is measured at each point. The resulting **compaction curve** is approximately parabolic and has a well-defined peak:

| Quantity | Symbol | Units | Meaning |
|---|---|---|---|
| **Optimum Moisture Content** | OMC | % | Water content at maximum dry unit weight |
| **Maximum Dry Density** | MDD | kN/m³ | Peak dry unit weight on the compaction curve |

---

### How soil type affects compaction behavior

Different soil families behave quite differently under compaction:

**Sandy soils (coarse-grained)**
- Low OMC — sand needs relatively little water to lubricate particle contact
- Relatively high MDD — dense particle packing is achievable at low water contents
- Compaction curve is broad and relatively flat
- Low fines content, low or non-plastic Atterberg limits

**Silty soils (intermediate)**
- Moderate OMC and MDD
- More sensitive to water content than sand
- Moderate fines content, low to moderate plasticity

**Clayey soils (fine-grained)**
- High OMC — clay platelets require substantial water to rearrange
- Lower MDD — excess water displaces particles that could otherwise be packed together
- High fines content, high liquid limit and plasticity index
- Compaction curve is narrow and sharply peaked

These systematic differences form the physical basis for the clustering you will perform in this assignment.

## Part 1: Load the Data

Run the cell below to load both CSV files. You do not need to modify this cell.

- **`curves`** — raw Proctor compaction points, one row per measurement
  - `test_id`, `point_id`, `moisture_content_pct`, `dry_unit_weight_kN_m3`
- **`features`** — one summary row per test, with grain-size and Atterberg limit data
  - `test_id`, `omc_pct`, `mdd_kN_m3`, `gravel_pct`, `sand_pct`, `fines_pct`,
    `liquid_limit_pct`, `plastic_limit_pct`, `plasticity_index_pct`, `specific_gravity`

> **Note:** `omc_pct` and `mdd_kN_m3` in the features file are provided for reference. In Question 2 you will recover these values yourself from the raw curve data.

In [ ]:
# Run this cell — do not modify
curves   = pd.read_csv("proctor_curves.csv")
features = pd.read_csv("soil_compaction_features.csv")

print("Proctor curves — shape:", curves.shape)
print("Feature summary — shape:", features.shape)

print("\nProctor curves (first 5 rows):")
display(curves.head())

print("\nFeature summary (first 5 rows):")
display(features.head())

## Question 1: Fit a Polynomial to a Compaction Curve

Taking the raw data maximum as the MDD is straightforward, but it is sensitive to measurement noise: if one test point happens to be slightly high or low, the estimated peak shifts. Fitting a **polynomial** to the compaction data and then finding its mathematical peak is more robust and is standard practice in automated Proctor data reduction.

Write a function named `fitProctorCurve` that fits a polynomial of a given degree to a set of compaction measurements and returns the resulting polynomial as a callable object.

### Input arguments

| Argument | Type | Description |
|---|---|---|
| `moisture_pcts` | array-like | Moisture content values for each compaction point (%) |
| `dry_unit_weights` | array-like | Corresponding dry unit weights (kN/m³) |
| `degree` | int | Degree of the fitting polynomial (default `2`) |

### Output

| Return value | Type | Description |
|---|---|---|
| `poly_fn` | `numpy.poly1d` | Polynomial function fitted to the compaction data |

### Hints
- Use `np.polyfit` to obtain the polynomial coefficients.
- Wrap the coefficients with `np.poly1d` to get a callable function.
- This is the same workflow as `fitCompactionCurve` from the *Soil Compaction* notebook — the difference here is that you will apply it to many tests automatically.

In [ ]:
def fitProctorCurve(moisture_pcts, dry_unit_weights, degree=2):
    moisture_pcts    = np.array(moisture_pcts)
    dry_unit_weights = np.array(dry_unit_weights)
    coeffs   = np.polyfit(moisture_pcts, dry_unit_weights, degree)  # SOLUTION
    poly_fn  = np.poly1d(coeffs)                                    # SOLUTION
    return poly_fn

### Helper Visualization

Run the cell below after implementing `fitProctorCurve` to see the fitted curve for test **T01**. A good fit will pass smoothly through the data points and show a clear peak.

In [ ]:
# Helper: plot one fitted compaction curve
example_id = "T01"

one_curve = curves[curves["test_id"] == example_id].sort_values("moisture_content_pct")
w  = one_curve["moisture_content_pct"].to_numpy()
gd = one_curve["dry_unit_weight_kN_m3"].to_numpy()

poly_fn_example = fitProctorCurve(w, gd, degree=2)
w_smooth = np.linspace(w.min(), w.max(), 300)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(w, gd, color="black", zorder=5, s=50, label="Measured points")
ax.plot(w_smooth, poly_fn_example(w_smooth), color="royalblue", linewidth=2,
        label="Degree-2 polynomial fit")
ax.set_xlabel("Moisture Content (%)")
ax.set_ylabel("Dry Unit Weight (kN/m³)")
ax.set_title(f"Proctor compaction curve — Test {example_id}")
ax.legend()
ax.grid(True, linestyle=":", linewidth=0.8)
plt.tight_layout()
plt.show()

#### Testing Cell:

In [ ]:
# Sanity-check fitProctorCurve against a known dataset
_w_test  = [8, 10, 12, 14, 16, 18]
_gd_test = [16.5, 17.8, 18.6, 18.9, 18.4, 17.6]
q1 = fitProctorCurve(_w_test, _gd_test, degree=2)

assert get_hash(round(float(q1(9)),  1)) == "1b7f864ac7a422c1a24c720d543e7e0f", \
    "fitProctorCurve: unexpected value at w=9"
assert get_hash(round(float(q1(10)), 1)) == "1383c2b19889c144c1c67203d8f7d201", \
    "fitProctorCurve: unexpected value at w=10"
assert get_hash(round(float(q1(13)), 1)) == "4652e04e446dd7e02c704d4189292a1c", \
    "fitProctorCurve: unexpected value at w=13"
print("fitProctorCurve ✓ all checks passed")

## Question 2: Extract OMC and MDD from the Fitted Curve

Now that you can fit a polynomial to a compaction curve, you need to find its **peak** — the point where dry unit weight is maximized. This peak defines the OMC and MDD.

For a degree-2 polynomial, the peak is a single well-defined maximum. A practical and numerically robust approach is to evaluate the polynomial at many closely spaced moisture content values and find which one gives the highest dry unit weight.

Write a function named `computeOMC_MDD` that returns the OMC and MDD from a fitted polynomial.

### Input arguments

| Argument | Type | Description |
|---|---|---|
| `poly_fn` | `numpy.poly1d` | Polynomial function from `fitProctorCurve` |
| `w_min` | float | Lower bound of the moisture content range (%) |
| `w_max` | float | Upper bound of the moisture content range (%) |

### Output

| Return value | Type | Description |
|---|---|---|
| `omc` | float | Optimum moisture content (%) |
| `mdd` | float | Maximum dry unit weight (kN/m³) |

### Hints
- Create a dense array of moisture contents with `np.linspace(w_min, w_max, 1000)`.
- Evaluate the polynomial at each point with `poly_fn(w_dense)`.
- Use `np.argmax` to locate the index of the maximum dry unit weight.
- Return `omc` and `mdd` as Python `float` values (cast with `float(...)`).

In [ ]:
def computeOMC_MDD(poly_fn, w_min, w_max):
    w_dense  = np.linspace(w_min, w_max, 1000)       # SOLUTION
    gd_dense = poly_fn(w_dense)                       # SOLUTION
    idx_max  = np.argmax(gd_dense)                    # SOLUTION
    omc = float(w_dense[idx_max])                     # SOLUTION
    mdd = float(gd_dense[idx_max])                    # SOLUTION
    return omc, mdd

### Helper Visualization

The cell below fits curves for six representative tests — two from each expected soil family — and marks the OMC (vertical dashed red line) and MDD (horizontal dashed green line). Notice how OMC increases and MDD decreases as you move from sandy to clayey soils.

In [ ]:
# Helper: show fitted curves with OMC and MDD for six tests
sample_tests = ["T01", "T03", "T15", "T22", "T27", "T33"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, tid in enumerate(sample_tests):
    one = curves[curves["test_id"] == tid].sort_values("moisture_content_pct")
    w   = one["moisture_content_pct"].to_numpy()
    gd  = one["dry_unit_weight_kN_m3"].to_numpy()

    poly = fitProctorCurve(w, gd, degree=2)
    omc, mdd = computeOMC_MDD(poly, w.min(), w.max())

    w_smooth = np.linspace(w.min(), w.max(), 300)
    axes[i].scatter(w, gd, color="black", s=20, zorder=5)
    axes[i].plot(w_smooth, poly(w_smooth), color="royalblue")
    axes[i].axvline(omc, color="red",   linestyle="--",
                    label=f"OMC = {omc:.1f}%")
    axes[i].axhline(mdd, color="green", linestyle="--",
                    label=f"MDD = {mdd:.2f} kN/m³")
    axes[i].set_title(f"Test {tid}")
    axes[i].set_xlabel("w (%)")
    axes[i].set_ylabel("γd (kN/m³)")
    axes[i].legend(fontsize=7)
    axes[i].grid(True, linestyle=":", linewidth=0.6)

plt.suptitle("Fitted compaction curves with OMC and MDD", fontsize=13)
plt.tight_layout()
plt.show()

### Recover OMC and MDD for all 36 tests

Run the cell below to apply your two functions to every test in the dataset and compare the curve-derived OMC/MDD against the pre-computed values in the features file. Small differences are expected — the feature file values were generated from a finer polynomial evaluation, so discrepancies of less than **0.5%** for OMC and **0.05 kN/m³** for MDD are normal.

In [ ]:
# Helper: apply fitProctorCurve + computeOMC_MDD to all tests
records = []
for tid, grp in curves.groupby("test_id"):
    grp = grp.sort_values("moisture_content_pct")
    w   = grp["moisture_content_pct"].to_numpy()
    gd  = grp["dry_unit_weight_kN_m3"].to_numpy()
    poly = fitProctorCurve(w, gd, degree=2)
    omc, mdd = computeOMC_MDD(poly, w.min(), w.max())
    records.append({"test_id": tid, "omc_curve": round(omc, 2), "mdd_curve": round(mdd, 3)})

curve_summary = pd.DataFrame(records)
comparison = features[["test_id", "omc_pct", "mdd_kN_m3"]].merge(curve_summary, on="test_id")
comparison["omc_diff"] = (comparison["omc_curve"] - comparison["omc_pct"]).round(3)
comparison["mdd_diff"] = (comparison["mdd_curve"] - comparison["mdd_kN_m3"]).round(4)

print("=== OMC and MDD: feature file vs. curve-derived ===")
display(comparison)
print(f"\nMax |OMC diff|: {comparison['omc_diff'].abs().max():.3f} %")
print(f"Max |MDD diff|: {comparison['mdd_diff'].abs().max():.4f} kN/m³")

#### Testing Cell:

In [ ]:
# Check computeOMC_MDD on a known polynomial
_w_test  = [8, 10, 12, 14, 16, 18]
_gd_test = [16.5, 17.8, 18.6, 18.9, 18.4, 17.6]
_poly    = fitProctorCurve(_w_test, _gd_test, degree=2)
q2_omc, q2_mdd = computeOMC_MDD(_poly, min(_w_test), max(_w_test))

assert abs(q2_omc - 13.8) < 0.5, \
    f"computeOMC_MDD: OMC = {q2_omc:.2f}, expected near 13.8"
assert abs(q2_mdd - 18.9) < 0.2, \
    f"computeOMC_MDD: MDD = {q2_mdd:.3f}, expected near 18.9"
print(f"computeOMC_MDD ✓  OMC = {q2_omc:.2f}%,  MDD = {q2_mdd:.3f} kN/m³")

## Background: Building a Feature Matrix for Clustering

K-means treats each observation as a point in a multi-dimensional feature space and groups observations by proximity to cluster centroids. For this to work well:

1. **Each row** of the feature matrix must represent one observation (one compaction test).
2. **Each column** must represent one numerical feature.
3. The features must be on a **comparable scale** — otherwise a variable measured in the tens (e.g. OMC ≈ 10–25%) will dominate a variable measured in the ones (e.g. specific gravity ≈ 2.67), and the clustering will effectively ignore the smaller-scale variable.

### Whitening

SciPy’s `scipy.cluster.vq.whiten` rescales each feature by dividing it by its sample standard deviation, so that every whitened feature has a standard deviation of 1.0. The clustering then treats all features equally in Euclidean distance calculations.

$$\tilde{x}_j = \frac{x_j}{\sigma_j}$$

where $x_j$ is the original feature column and $\sigma_j$ is its sample standard deviation.

> **Important:** `whiten` does **not** center the data (i.e. it does not subtract the mean). It only scales by the standard deviation. The whitened features will still be offset from zero if the original features were.

## Question 3: Build and Whiten the Feature Matrix

Write a function named `buildFeatureMatrix` that selects a set of numerical columns from the feature table, drops any rows with missing values, converts the result to a NumPy array, and returns both the raw array and the whitened version.

### Input arguments

| Argument | Type | Description |
|---|---|---|
| `features_df` | `pandas.DataFrame` | The features table (one row per test) |
| `feature_cols` | list of str | Column names to include in the feature matrix |

### Output

| Return value | Type | Description |
|---|---|---|
| `X` | `numpy.ndarray`, shape (n, p) | Raw feature matrix |
| `X_white` | `numpy.ndarray`, shape (n, p) | Whitened feature matrix |

### Hints
- Use `features_df[feature_cols].dropna()` to select and clean the data.
- Call `.to_numpy()` to convert to a NumPy array.
- Apply `whiten(X)` from `scipy.cluster.vq` to get `X_white`.

In [ ]:
def buildFeatureMatrix(features_df, feature_cols):
    X_df    = features_df[feature_cols].dropna()  # SOLUTION
    X       = X_df.to_numpy()                     # SOLUTION
    X_white = whiten(X)                           # SOLUTION
    return X, X_white

### Choosing features and inspecting the whitened matrix

Run the cell below to build the feature matrix with a default set of four columns. After whitening, the standard deviation of each column should be **exactly 1.0**. You can add or remove columns from `feature_cols` to explore how your choice affects the clustering in later questions.

In [ ]:
# Define the feature columns to use for clustering
# You may add or remove columns, but start with these four
feature_cols = [
    "omc_pct",
    "mdd_kN_m3",
    "fines_pct",
    "plasticity_index_pct",
]

X, X_white = buildFeatureMatrix(features, feature_cols)

print(f"Feature matrix shape: {X.shape}  (rows = tests, cols = features)")

print("\n=== Original feature statistics ===")
display(pd.DataFrame(X, columns=feature_cols).describe().round(2))

print("\n=== Whitened feature statistics (std should be ≈ 1 for each column) ===")
display(pd.DataFrame(X_white, columns=[c + "_w" for c in feature_cols]).describe().round(2))

#### Testing Cell:

In [ ]:
# Check shape and whitening
_feature_cols = ["omc_pct", "mdd_kN_m3", "fines_pct", "plasticity_index_pct"]
_X, _X_white = buildFeatureMatrix(features, _feature_cols)

assert _X.shape == (36, 4), \
    f"buildFeatureMatrix: expected shape (36, 4), got {_X.shape}"

col_stds = _X_white.std(axis=0)
assert all(abs(s - 1.0) < 1e-6 for s in col_stds), \
    f"buildFeatureMatrix: whitened column stds should all be 1.0, got {col_stds}"

print("buildFeatureMatrix ✓  shape and whitening checks passed")

## Background: K-means Clustering

K-means partitions observations into $k$ groups by iteratively:

1. Assigning each observation to the nearest centroid (in Euclidean distance).
2. Recomputing each centroid as the mean of its assigned observations.

The algorithm converges when assignments stop changing. SciPy provides two implementations:

| Function | Returns | Notes |
|---|---|---|
| `kmeans(obs, k_or_guess)` | `(centroids, distortion)` | Returns only the centroids and total distortion |
| `kmeans2(data, k)` | `(centroids, labels)` | Also returns per-observation cluster labels |

We will use **`kmeans2`** because it returns both centroids and labels in one call.

### Initialization

The final clusters can depend on where the algorithm starts. `kmeans2` offers several `minit` options:

| `minit` value | Description |
|---|---|
| `"random"` | Randomly chosen initial centroids |
| `"points"` | Randomly chosen observations as initial centroids |
| `"++"` | K-means++ — spreads initial centroids out deliberately; more robust |

We will use `"++"` and fix the random seed so that results are reproducible.

### Why $k = 3$?

We are looking for three broad soil families: sandy, silty, and clayey. In a real investigation you might use the **elbow method** or the **silhouette score** to choose $k$ empirically. Here we set $k = 3$ because we have prior geotechnical knowledge about how many groups to expect. In the reflection section you will examine what happens when $k$ is different.

## Question 4: Cluster the Compaction Tests

Write a function named `clusterSoils` that applies `kmeans2` to a whitened feature matrix and returns the cluster centroids and per-observation labels.

### Input arguments

| Argument | Type | Description |
|---|---|---|
| `X_white` | `numpy.ndarray` | Whitened feature matrix from `buildFeatureMatrix` |
| `k` | int | Number of clusters |
| `seed` | int | Random seed for reproducibility (default `0`) |

### Output

| Return value | Type | Description |
|---|---|---|
| `centroids_white` | `numpy.ndarray`, shape (k, p) | Cluster centroids in whitened space |
| `labels` | `numpy.ndarray`, shape (n,) | Cluster index (0 to k-1) for each observation |

### Hints
- Call `kmeans2(X_white, k, minit="++", rng=seed)`.
- `kmeans2` returns `(centroids, labels)` directly.

In [ ]:
def clusterSoils(X_white, k, seed=0):
    centroids_white, labels = kmeans2(X_white, k, minit="++", rng=seed)  # SOLUTION
    return centroids_white, labels

### Run clustering and visualize the result

Run the cell below to cluster the 36 tests into three groups and see how they separate in two different 2D projections of the feature space. If the clustering is working well, the three groups should be visually distinct in at least one of the two plots.

In [ ]:
# Cluster the soils and inspect the results
k = 3
centroids_white, labels = clusterSoils(X_white, k=k, seed=0)

df_clustered = features.dropna(subset=feature_cols).copy()
df_clustered["cluster"] = labels

print("=== Cluster counts ===")
display(df_clustered["cluster"].value_counts().sort_index().rename("n_tests"))

colors = ["tab:blue", "tab:orange", "tab:green"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: OMC vs MDD
for cl in range(k):
    mask = df_clustered["cluster"] == cl
    axes[0].scatter(df_clustered.loc[mask, "omc_pct"],
                    df_clustered.loc[mask, "mdd_kN_m3"],
                    color=colors[cl], label=f"Cluster {cl}", s=55, alpha=0.85)
axes[0].set_xlabel("OMC (%)")
axes[0].set_ylabel("MDD (kN/m³)")
axes[0].set_title("K-means clusters: OMC vs MDD")
axes[0].legend()
axes[0].grid(True, linestyle=":", linewidth=0.8)

# Plot 2: Fines vs Plasticity Index
for cl in range(k):
    mask = df_clustered["cluster"] == cl
    axes[1].scatter(df_clustered.loc[mask, "fines_pct"],
                    df_clustered.loc[mask, "plasticity_index_pct"],
                    color=colors[cl], label=f"Cluster {cl}", s=55, alpha=0.85)
axes[1].set_xlabel("Fines Content (%)")
axes[1].set_ylabel("Plasticity Index (%)")
axes[1].set_title("K-means clusters: Fines vs Plasticity Index")
axes[1].legend()
axes[1].grid(True, linestyle=":", linewidth=0.8)

plt.suptitle("K-means clustering of compaction tests  (k = 3)", fontsize=13)
plt.tight_layout()
plt.show()

### Confirming labels with `vq`

SciPy also provides `vq` (vector quantization), which assigns each observation to the nearest centroid after clustering. When you call it with the centroids already returned by `kmeans2`, the result should match the `labels` array exactly. This is a useful sanity check and also shows the complete SciPy clustering workflow.

In [ ]:
# Confirm that vq produces the same assignments as kmeans2
vq_codes, vq_distances = vq(X_white, centroids_white)

match = np.array_equal(labels, vq_codes)
print(f"kmeans2 labels and vq codes match: {match}")

print("\nDistance to assigned centroid (first 10 tests):")
display(pd.DataFrame({
    "test_id": df_clustered["test_id"].values[:10],
    "cluster": vq_codes[:10],
    "dist_to_centroid": vq_distances[:10].round(3)
}))

#### Testing Cell:

In [ ]:
# Check clusterSoils output shapes and label validity
_cw, _labs = clusterSoils(X_white, k=3, seed=0)

assert _cw.shape == (3, len(feature_cols)), \
    f"clusterSoils: centroid shape should be (3, {len(feature_cols)}), got {_cw.shape}"
assert len(_labs) == X_white.shape[0], \
    f"clusterSoils: labels length should be {X_white.shape[0]}, got {len(_labs)}"
assert set(_labs) == {0, 1, 2}, \
    f"clusterSoils: expected labels {{0,1,2}}, got {set(_labs)}"
print("clusterSoils ✓  shape and label checks passed")

## Background: Interpreting Cluster Centroids

K-means assigns each test to a numbered cluster (0, 1, 2), but it has no knowledge of soil science. To translate cluster numbers into soil-type labels, you need to examine the **centroid values** in engineering units and reason about what type of soil they most likely represent.

### Converting centroids back to engineering units

The centroids returned by `kmeans2` are in **whitened space** — each feature has been divided by its sample standard deviation. To interpret the centroids physically, multiply each column of the centroid matrix by the corresponding standard deviation of the original feature:

$$\text{centroid in original units} = \tilde{c}_j \times \sigma_j$$

where $\tilde{c}_j$ is the centroid value in whitened space and $\sigma_j$ is the standard deviation of feature $j$ computed from $X$ (not $X_{\text{white}}$).

> **Note:** This is an approximation. Because `whiten` does not center the data, the centroids in original units are relative to the **origin**, not to the feature means. For interpretation purposes, the important quantity is how the centroids **compare to each other**, not their absolute values.

### What to look for

Use the table below as a reference when examining the centroid values:

| Soil type | OMC | MDD | Fines | Plasticity Index |
|---|---|---|---|---|
| Sandy | Low (≈10–12%) | High (≈18–19 kN/m³) | Low (<25%) | Low (<8%) |
| Silty | Moderate (≈14–16%) | Moderate (≈17–18 kN/m³) | Moderate (40–60%) | Low–moderate (<12%) |
| Clayey | High (>18%) | Low (<17 kN/m³) | High (>60%) | High (>15%) |


## Question 5: Map Cluster Numbers to Soil Groups

Study the centroid table printed by the helper cell below, then assign each cluster number to the soil group you believe it most likely represents.

Create a dictionary named `q5` that maps each cluster index (0, 1, 2) to one of the following **exact strings**: `"Sandy"`, `"Silty"`, or `"Clayey"`.

Each soil group must be assigned to **exactly one** cluster.

### Example (do not copy directly — use your own interpretation)

```python
q5 = {
    0: "Clayey",
    1: "Sandy",
    2: "Silty",
}
```

### Hints
- Run the centroid helper cell below before filling in your answer.
- The cluster with the **lowest OMC and highest MDD** is most likely sandy.
- The cluster with the **highest OMC and lowest MDD** is most likely clayey.
- If two clusters are ambiguous, use fines content and plasticity index to distinguish them.

### Centroid helper

Run this cell to display the centroid table in approximate engineering units. Use the values to guide your answer in the cell below.

In [ ]:
# Helper: centroid table in approximate engineering units
feature_stds = X.std(axis=0)
centroids_eng = centroids_white * feature_stds

centroid_df = pd.DataFrame(centroids_eng, columns=feature_cols)
centroid_df.index.name = "Cluster"

print("=== Cluster centroids (approximate engineering units) ===")
print("(Values are centroid coordinates in whitened space * std; ")
print(" compare clusters relative to each other, not to absolute benchmarks.)")
display(centroid_df.round(3))

In [ ]:
# TODO: replace the values below with your interpretation
q5 = {
    0: "Sandy",   # SOLUTION
    1: "Silty",   # SOLUTION
    2: "Clayey",  # SOLUTION
}

### Labeled scatter plot

Once you have filled in `q5`, run the helper cell below to see your predicted soil groups plotted in OMC–MDD space. The clusters should align with the engineering expectations described in the background section: sandy soils at low OMC and high MDD, clayey soils at high OMC and low MDD.

In [ ]:
# Helper: scatter plot labeled by your predicted soil group
df_labeled = df_clustered.copy()
df_labeled["soil_group"] = df_labeled["cluster"].map(q5)

group_colors = {"Sandy": "tab:blue", "Silty": "tab:orange", "Clayey": "tab:green"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for group, grp_df in df_labeled.groupby("soil_group"):
    axes[0].scatter(grp_df["omc_pct"], grp_df["mdd_kN_m3"],
                    color=group_colors[group], label=group, s=55, alpha=0.85)
axes[0].set_xlabel("OMC (%)")
axes[0].set_ylabel("MDD (kN/m³)")
axes[0].set_title("Predicted soil groups: OMC vs MDD")
axes[0].legend()
axes[0].grid(True, linestyle=":", linewidth=0.8)

for group, grp_df in df_labeled.groupby("soil_group"):
    axes[1].scatter(grp_df["fines_pct"], grp_df["plasticity_index_pct"],
                    color=group_colors[group], label=group, s=55, alpha=0.85)
axes[1].set_xlabel("Fines Content (%)")
axes[1].set_ylabel("Plasticity Index (%)")
axes[1].set_title("Predicted soil groups: Fines vs Plasticity Index")
axes[1].legend()
axes[1].grid(True, linestyle=":", linewidth=0.8)

plt.suptitle("Predicted soil group assignments", fontsize=13)
plt.tight_layout()
plt.show()

print("\n=== Tests per predicted soil group ===")
display(df_labeled["soil_group"].value_counts().rename("n_tests"))

### Compare against the instructor answer key

The file `instructor_answer_key.csv` contains the true soil group for each test. Run the cell below to see how well your cluster-to-group mapping recovered the original labels.

In [ ]:
# Helper: compare predictions to the answer key
answer_key = pd.read_csv("instructor_answer_key.csv")

comparison = df_labeled[["test_id", "soil_group"]].merge(
    answer_key, on="test_id"
)

# Rename answer key column if needed
true_col = [c for c in answer_key.columns if c != "test_id"][0]
comparison = comparison.rename(columns={true_col: "true_group"})

comparison["correct"] = comparison["soil_group"] == comparison["true_group"]
accuracy = comparison["correct"].mean()

print(f"Accuracy: {accuracy:.1%}  ({comparison['correct'].sum()}/{len(comparison)} tests correct)")
display(comparison)

#### Testing Cell:

In [ ]:
# Check that q5 is a valid mapping with the correct soil-group names
_valid_groups = {"Sandy", "Silty", "Clayey"}

assert isinstance(q5, dict), "q5 should be a dictionary"
assert set(q5.keys()) == {0, 1, 2}, \
    f"q5 should have keys {{0, 1, 2}}, got {set(q5.keys())}"
assert set(q5.values()) == _valid_groups, \
    f"q5 values must be {_valid_groups}, got {set(q5.values())}. Check spelling."
print("q5 ✓  format and label checks passed")

## Reflection

Answer the following questions in the markdown cells provided. There are no autograded tests for this section; your answers will be reviewed manually.

### Reflection 1

Re-run the clustering using **only** `omc_pct` and `mdd_kN_m3` (remove `fines_pct` and `plasticity_index_pct` from `feature_cols`). Compare the resulting accuracy against the full four-feature model.

**Which feature set performed better, and why might adding grain-size and Atterberg limit data help or hurt the clustering?**

**Your answer:**

*[double-click to edit]*

### Reflection 2

Re-run `clusterSoils` with `k = 2` and `k = 4`. Describe what you observe in the resulting scatter plots.

**What happens to the silty group when $k = 2$? Does increasing to $k = 4$ produce a geotechnically meaningful fourth group, or does it split an existing group arbitrarily?**

**Your answer:**

*[double-click to edit]*

In [ ]:
# Exploration cell: try k=2 and k=4
# Re-use buildFeatureMatrix and clusterSoils from above

for k_test in [2, 4]:
    c_w, lbs = clusterSoils(X_white, k=k_test, seed=0)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(features.dropna(subset=feature_cols)["omc_pct"],
               features.dropna(subset=feature_cols)["mdd_kN_m3"],
               c=lbs, cmap="tab10", s=55, alpha=0.85)
    ax.set_xlabel("OMC (%)")
    ax.set_ylabel("MDD (kN/m³)")
    ax.set_title(f"K-means with k = {k_test}")
    ax.grid(True, linestyle=":", linewidth=0.8)
    plt.tight_layout()
    plt.show()

### Reflection 3

K-means assumes that clusters are roughly **spherical and equal in size** in the feature space, and it uses Euclidean distance as its similarity measure.

**Give a concrete geotechnical scenario where these assumptions might cause K-means to incorrectly group two tests that a geotechnical engineer would classify differently. Use specific feature values in your example.**

**Your answer:**

*[double-click to edit]*